# Gold Price Prediction — Advanced ML Pipeline

This notebook walks through the full advanced workflow:

1. Load and merge Yahoo Finance + FRED macro data  
2. Feature engineering (lags, returns, technical indicators, macro variables)  
3. Train/test split respecting temporal order  
4. Model comparison with **TimeSeriesSplit** cross-validation  
5. **Hyperparameter tuning** with `RandomizedSearchCV` + `TimeSeriesSplit`  
6. Feature selection using feature importances  
7. Final evaluation on the held-out test set  
8. Error analysis and visualisations  

> **Prerequisites:** Run the data download scripts first, or skip macro data  
> ```bash
> pip install -r requirements.txt
> python scripts/download_market_data.py
> python scripts/download_fred_data.py  # optional, needs FRED_API_KEY
> ```

In [ ]:
import sys, os
# Make sure the project root is in the path when running from the notebooks/ folder
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1 — Load and inspect data

In [ ]:
DATA_DIR = os.path.join('..', 'data')
yahoo_path = os.path.join(DATA_DIR, 'yahoo_market_data.csv')
fred_path  = os.path.join(DATA_DIR, 'fred_macro_data.csv')

raw = pd.read_csv(yahoo_path, parse_dates=['Date'], index_col='Date')
raw.sort_index(inplace=True)

if os.path.exists(fred_path):
    fred = pd.read_csv(fred_path, parse_dates=['Date'], index_col='Date')
    fred = fred.reindex(raw.index, method='ffill')
    raw = pd.concat([raw, fred], axis=1)
    print('FRED macro features merged.')

print(f'Shape: {raw.shape}')
raw.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
pairs = [
    ('gold_Close',   'Gold Close Price (USD)'),
    ('sp500_Close',  'S&P 500 Close'),
    ('dxy_Close',    'DXY (US Dollar Index)'),
    ('nasdaq_Close', 'Nasdaq Close'),
]
for ax, (col, title) in zip(axes.flat, pairs):
    if col in raw.columns:
        raw[col].plot(ax=ax, linewidth=0.8)
        ax.set_title(title)
        ax.set_xlabel('')
plt.tight_layout()
plt.show()

## 2 — Feature engineering

In [ ]:
from src.features import build_features

TASK    = 'regression'   # change to 'classification' for up/down prediction
HORIZON = 1              # number of trading days ahead to predict

df = build_features(raw, task=TASK, horizon=HORIZON)
feature_cols = [c for c in df.columns if c != 'target']

print(f'Feature matrix: {df.shape}')
print(f'Number of features: {len(feature_cols)}')
print(f'Date range: {df.index[0].date()} → {df.index[-1].date()}')
df[['gold_Close', 'target'] + feature_cols[:5]].head()

In [ ]:
# Correlation heatmap — top 20 features most correlated with target
corr = df[feature_cols + ['target']].corr()['target'].drop('target').abs()
top20 = corr.nlargest(20).index.tolist()

plt.figure(figsize=(10, 6))
corr[top20].sort_values().plot(kind='barh')
plt.title('Top 20 Features — Absolute Correlation with Target')
plt.xlabel('|Pearson correlation|')
plt.tight_layout()
plt.show()

## 3 — Train / test split

In [ ]:
TEST_RATIO = 0.2
split_idx  = int(len(df) * (1 - TEST_RATIO))

train_df = df.iloc[:split_idx]
test_df  = df.iloc[split_idx:]

X_train, y_train = train_df[feature_cols], train_df['target']
X_test,  y_test  = test_df[feature_cols],  test_df['target']

print(f'Train: {X_train.shape[0]} rows  ({X_train.index[0].date()} → {X_train.index[-1].date()})')
print(f'Test:  {X_test.shape[0]}  rows  ({X_test.index[0].date()}  → {X_test.index[-1].date()})')

## 4 — Model comparison with TimeSeriesSplit cross-validation

In [ ]:
from src.models import get_regression_models, get_classification_models
from src.validation import compare_models_cv

N_CV_SPLITS = 5

if TASK == 'regression':
    models = get_regression_models()
else:
    models = get_classification_models()

print(f'Comparing {len(models)} models with {N_CV_SPLITS}-fold TimeSeriesSplit ...')
cv_results = compare_models_cv(models, X_train, y_train, n_splits=N_CV_SPLITS, task=TASK)
cv_results

In [ ]:
# Visualise CV results
primary_col = 'MAE_mean' if TASK == 'regression' else 'Accuracy_mean'
err_col     = 'MAE_std'  if TASK == 'regression' else 'Accuracy_std'

cv_sorted = cv_results.sort_values(primary_col, ascending=(TASK == 'regression'))

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(
    cv_sorted.index,
    cv_sorted[primary_col],
    xerr=cv_sorted[err_col],
    capsize=4,
    color='steelblue',
)
ax.set_xlabel(primary_col)
ax.set_title(f'TimeSeriesSplit CV — {primary_col} (lower is better for MAE)')
plt.tight_layout()
plt.show()

best_model_name = cv_sorted.index[0]
print(f'Best model by CV: {best_model_name}')

## 5 — Hyperparameter tuning

In [ ]:
from src.tuning import randomized_search, best_params_report, PARAM_GRIDS

# Use the best model found above
best_model = models[best_model_name]

class_name = type(best_model).__name__
if class_name in PARAM_GRIDS:
    print(f'Tuning {best_model_name} ({class_name}) ...')
    search = randomized_search(
        best_model,
        X_train, y_train,
        n_iter=20,          # increase for a thorough search
        n_splits=N_CV_SPLITS,
        task=TASK,
        random_state=42,
    )
    print('\nBest parameters:')
    print(search.best_params_)
    print('\nTop-10 parameter combinations:')
    display(best_params_report(search))
    tuned_model = search.best_estimator_
else:
    print(f'No default grid for {class_name} — skipping tuning.')
    tuned_model = best_model

## 6 — Feature selection

In [ ]:
from src.models import feature_importance_df, select_top_features

try:
    fi = feature_importance_df(tuned_model, feature_cols)
    top20_feats = fi.head(20)

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(top20_feats['feature'][::-1], top20_feats['importance'][::-1], color='teal')
    ax.set_xlabel('Feature Importance')
    ax.set_title('Top 20 Feature Importances')
    plt.tight_layout()
    plt.show()

    # Optionally re-train on top features only
    TOP_N = 30
    top_feats = select_top_features(tuned_model, feature_cols, top_n=TOP_N)
    print(f'Top {TOP_N} features selected: {top_feats[:5]} ...')

except ValueError as e:
    print(f'Feature importance not available: {e}')
    top_feats = feature_cols

## 7 — Final evaluation on the held-out test set

In [ ]:
from src.models import compare_on_test

# Compare tuned best model against all baselines on the test set
print('Evaluating all models on the held-out test set ...')
test_results = compare_on_test(
    models,
    X_train, y_train,
    X_test,  y_test,
    task=TASK,
)
test_results

In [ ]:
# Predictions vs actuals for the best model
from sklearn.metrics import mean_absolute_error

tuned_model.fit(X_train, y_train)
preds = tuned_model.predict(X_test)

plt.figure(figsize=(13, 4))
plt.plot(y_test.index, y_test.values, label='Actual', linewidth=1.0)
plt.plot(y_test.index, preds, label='Predicted', linewidth=1.0, alpha=0.8)
plt.title(f'{best_model_name} — Predicted vs Actual Gold Close (test set)')
plt.ylabel('Price (USD)')
plt.legend()
plt.tight_layout()
plt.show()

mae  = mean_absolute_error(y_test, preds)
rmse = np.sqrt(np.mean((y_test.values - preds) ** 2))
print(f'Test MAE:  {mae:.2f}  |  Test RMSE: {rmse:.2f}')

## 8 — Error analysis

In [ ]:
residuals = y_test.values - preds

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Residuals over time
axes[0].plot(y_test.index, residuals, linewidth=0.8, color='coral')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('Residuals over Time')
axes[0].set_ylabel('Residual (Actual − Predicted)')

# Residual histogram
axes[1].hist(residuals, bins=40, color='steelblue', edgecolor='white')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual')

# Predicted vs Actual scatter
axes[2].scatter(y_test.values, preds, alpha=0.3, s=10, color='teal')
lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
axes[2].plot(lims, lims, 'r--', linewidth=1)
axes[2].set_xlabel('Actual')
axes[2].set_ylabel('Predicted')
axes[2].set_title('Predicted vs Actual')

plt.tight_layout()
plt.show()

## 9 — Save the best model

In [ ]:
import joblib

model_dir = os.path.join('..', 'models')
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, f'{TASK}_best_model.joblib')

joblib.dump(tuned_model, model_path)
print(f'Model saved to {model_path}')

## Next steps

- **Increase `n_iter`** in `randomized_search` for a more thorough hyperparameter search  
- **Try `--top-features N`** in the pipeline CLI to restrict to the most predictive inputs  
- **LSTM / Transformer models** — for sequence-aware deep learning, see `tensorflow` or `pytorch`  
- **Backtesting** — simulate a simple trading strategy driven by model signals  
- **Periodic retraining** — add a cron job or GitHub Actions workflow to retrain weekly  
- **More FRED series** — add oil prices (DCOILWTICO), VIX (VIXCLS), or M2 money supply  